In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import neurokit2 as nk

neurokit2 0.2.13


In [ ]:
# File Paths
BASE = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV"

# --- Part 1 inputs/outputs ---
XDF_FOLDER = f"{BASE}/Data"
METADATA_CSV = f"{BASE}/Metadata.csv"
RAW_PATH = f"{BASE}/physio_raw.csv"

# --- Part 2 output ---
CLEAN_PATH = f"{BASE}/physio_clean.csv"

# --- Part 3 outputs ---
EPOCH_PATH = f"{BASE}/epoch_availability.csv"
EDA_SCREEN_PATH = f"{BASE}/eda_record_screens.csv"

# --- Part 4 inputs ---
REC_META = f"{BASE}/Eye-tracking/Recording ID Metadata.csv"
FIXATIONS = f"{BASE}/Eye-tracking/UAV Mapper/fixations.csv"
AOI_FIXATIONS = f"{BASE}/Eye-tracking/UAV Mapper/aoi_fixations.csv"
FACE_FIXATIONS = f"{BASE}/Eye-tracking/Face Mapper/fixations_on_face.csv"
SURVEY_CSV = f"{BASE}/mdmt_physio.csv"

CLOUD_ROOT = "Users/debbiehsu/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Documents/MHMUAV/R Code/Pupil Cloud"
PUPIL_DIR = f"{CLOUD_ROOT}/PupilClean"

# --- Part 4 output ---
FINAL_PATH = f"{BASE}/physio_eye_survey.csv"

In [4]:
# CSV Save & Load Helpers
# Add column types to CSV
CSV_DTYPES = {
    "ID": "int64",
    "unix_ns": "Int64",
    "DataSyncMarker": "float64",
    "accept": "boolean",
    "eda_ok": "boolean",
    "Fix_ID": "Int64",
    "Sac_ID": "Int64",
    "epoch": "int64",
    "n_peaks": "int64",
    "unix_start": "Int64"
}

CSV_DATES = ["dt_utc"]

def save_table(df, path):
    """Write a table to CSV without row-number column"""
    df.to_csv(path, index=False)
    print(f" wrote {len(df):,} rows -> {path}")
    return path

def load_table(path):
    """Read a CSV with the column types"""
    head = pd.read_csv(path, nrows=0)
    dtype = {c: t for c, t in CSV_DTYPES.items() if c in head.columns}
    for c in head.columns:
        if c.startswith("AOI_") or c in ("in_fixation", "in_saccade"):
            dtype[c] = "Int8"
    df = pd.read_csv(path, dtype=dtype)
    for c in CSV_DATES:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True, format="ISO8601") # Expect variation in time format, some have fractional seconds and some don't
        return df

In [ ]:
# Settings
# --- PPG ---
PPG_FS = 25 # Sampling rate
EPOCH_SEC = 10 # Quality check window length: 10s (Vorreuther et al. 2025)
EPOCH_LEN = PPG_FS * EPOCH_SEC # = 250 samples per epoch
PPG_CHANNELS = ("PPG_GRN_C", "PPG_GRN_IS")

# PPG Quality Validation: Four criterion gate (Orphanidou et al., 2015)
HR_MIN, HR_MAX = 30, 180 # Plausible heart rate, beats per minute
IBI_MAX_SEC = 3.0 # No gap between beats longer than 3s
IBI_RATIO_MAX = 2.2 # Longest gap < 2.2x the shortest
QUALITY_MIN = 0.86 # Template-match correlation floor

# --- EDA ---
EDA_FS = 15 # Sampling rate
EDA_SENTINEL = 10000.0 # EmotiBit device failure code, should be removed from the data
EDA_CEILING_uS = 40.0 # Sanity ceiling
EDA_TOL_SEC = 0.034 # Tolerance second, half of the 15ms spacing
EDA_MIN_RANGE_uS = 0.005 # Selected for this data only - p5-p95 dynamic range screen
EDA_ROLE_FILE = {"EDA_C": "EDA_C.csv", "EDA_IS": "EDA_IS.csv"}

# --- Study Design
TASK_LABELS = ("HH", "LL", "M") # Labels that represent task blocks
VIDEO_ORDERS = {
    1: ("LL", "HH", "M"), 2: ("LL", "M", "HH"), 3: ("HH", "LL", "M"),
    4: ("HH", "M", "LL"), 5: ("M", "LL", "HH"), 6: ("M", "HH", "LL"),
}
RESTART_FIXES = {25: 2, 46: 3} # Team -> which block number was restarted
TOLERANCE = 0.021 # Nearest-timestamp window for non-base streams


# Part 1 - XDF to 25Hz grid with UNIX Time

In [ ]:
# Export each XDF stream to its own CSV
def export_streams_to_csv(streams, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    SUFFIX = {"MD-V5-0001023": "_C", "MD-V6-0000405": "_IS"} # EmotiBit serial number for each role
    written = []

    for stream in streams:
        name = stream["info"]["name"][0]
        if name == "Keyboard" or "Neon" in name:
            continue

        source_list = stream["info"].get("source_id") or ["Unknown"]
        source_id = source_list[0] or "Unknown"
        suffix = SUFFIX.get(source_id, "")

        values = np.asarray(stream["time_series"])
        times = stream["time_stamps"]
        if len(values) == 0:
            continue

        if values.ndim == 1 or values.shape[1] == 1: # Single channel stream
            table = pd.DataFrame({"timestamp": times, "value": np.ravel(values)})
        else: # Multi-channel, e.g. DataSyncMarker
            cols = [f"channel_{i}" for i in range(values.shape[1])]
            table = pd.DataFrame(values, columns=cols)
            table.insert(0, "timestamp", times) # Put timestamp first

        table.to_csv(os.path.join(output_dir, f"{name}{suffix}.csv"), index=False)
        written.append(name + suffix)
    return written


In [5]:
# Turn the DataSyncMarker channel into reliability labels
def label_reliability(marker, video_condition):
    """Marker states (1=video, 2=no video, 3=skip) -> HH / LL / M / No Video"""
    order = VIDEO_ORDERS.get(video_condition)
    if order is None:
        return pd.Series([None] * len(marker), index = marker.index)

    labels = []
    block_count = 0
    previous = None

    for value in marker:
        state = int(value) if  pd.notna(value) else None
        if state != previous:
            if state == 1:
                block_count += 1
        previous = state

        if state == 1 and 1 <= block_count <= len(order):
            labels.append(order[block_count - 1])
        elif state == 2:
            labels.append("No Video")
        else:
            labels.append(None)
    return pd.Series(labels, index=marker.index)

def attach_datasync(grid, datasync_path, video_condition):
    ds = pd.read_csv(datasync_path).sort_values("timestamp").reset_index(drop=True)
    ds["timestamp"] = ds["timestamp"].round(3) # Millisecond rounding

    if "channel_0" in ds.columns: # Marker state stream
        grid = pd.merge_asof(grid, ds[["timestamp", "channel_0"]],
                             on="timestamp", direction="backward")
        grid = grid.rename(columns={"channel_0": "DataSyncMarker"})
        grid["Reliability"] = label_reliability(grid["DataSyncMarker"], video_condition)

    if "channel_1" in ds.columns: # UNIX time stream
        grid = pd.merge_asof(grid, ds[["timestamp", "channel_1"]],
                             on="timestamp", direction="backward")
        grid = grid.rename(columns={"channel_1": "DataSyncMarker_UNIX"})
    return grid


In [7]:
# Build the 25Hz grid
def tidy_column_names(grid):
    renames = {}
    for name in grid.columns:
        new = name [:-6] if name.endswith("_value") else name # Drop a trailing "_value"
        new = new.replace("TEMP1_", "TEMP_")
        if new != name:
            renames[name] = new
    return grid.rename(columns=renames)

def order_columns(grid):
    desired = ["ID", "Condition", "timestamp", "DataSyncMarker", "DataSyncMarker_UNIX", "Reliability", "PPG_GRN_C", "PPG_RED_C", "PPG_IR_C", "EDA_C", "TEMP_C", "THERM_C", "PPG_GRN_IS", "PPG_RED_IS", "PPG_IR_IS", "EDA_IS", "TEMP_IS", "THERM_IS",]
    front = [c for c in desired if c in grid.columns]
    rest = [c for c in grid.columns if c not in front]
    return grid[front + rest]

def merge_raw(input_dir, team_id, condition, video_condition, tolerance = TOLERANCE):
    all_files = glob.glob(os.path.join(input_dir, "*.csv")) # All csv files in input directory
    base_file = None
    datasync_file = None
    other_files = []

    for path in all_files:
        name = os.path.basename(path) # Filename without the folders
        if "PPG_GRN" in name and name.endswith("_C.csv"):
            base_file = path # The 25Hz clock we build on
        elif "DataSyncMarker" in name:
            datasync_file = path
        elif "Neon" in name or "Keyboard" in name:
            continue
        else:
            other_files.append(path)

    if base_file is None:
        raise FileNotFoundError(f"No PPG_GRN_C.csv in {input_dir}")

    base = pd.read_csv(base_file).sort_values("timestamp").reset_index(drop=True)
    base["timestamp"] = base["timestamp"].round(3)

    grid = pd.DataFrame({"ID": team_id, "Condition": condition,
                         "timestamp": base["timestamp"]}) # Scalars are repeated down the column???

    if datasync_file is not None:
        grid = attach_datasync(grid, datasync_file, video_condition)
    grid["PPG_GRN_C"] = base["value"].to_numpy()

    for path in sorted(other_files):
        stem = os.path.basename(path)[:-4] # Filename minus the ".csv"
        stream = pd.read_csv(path)
        if "timestamp" not in stream.columns:
            continue
        stream["timestamp"] = stream["timestamp"].round(3)






NameError: name 'TOLERANCE' is not defined

In [ ]:
()